In [1]:
import json
import os

from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.spice.spice import Spice
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer

In [2]:
RAW_DATASET_PATH = 'E:/Research/simplify-me/simplify_me_dataset/meta.json'
GEN_CAPTION_FOLDER = 'E:/Research/gen-captions/gen-captions'
SCORED_CAPTION_FOLDER = 'E:/Research/gen-captions/gen-captions/scored'

In [3]:
with open(RAW_DATASET_PATH, 'r') as f:
    raw_dataset = json.load(f)

raw_test_dataset = raw_dataset['test']

In [4]:
print(len(raw_test_dataset))

13491


In [5]:
generated_files = os.listdir(SCORED_CAPTION_FOLDER)

print(generated_files)
print(len(generated_files))

['gemma3-3-epoch-1000-trained.json', 'gemma3-3-epoch-10000-trained.json', 'gemma3-3-epoch-full-trained.json', 'gemma3-base.json', 'Llama-3.2-11B-Vision-base.json', 'Llama-32-11B-Vision-1000-trained.json', 'Llama-32-11B-Vision-10000-trained.json', 'Llama-32-11B-Vision-full-trained.json', 'Qwen2-VL-2B-Instruct-1000-trained.json', 'Qwen2-VL-2B-Instruct-10000-trained.json', 'Qwen2-VL-2B-Instruct-base.json', 'Qwen2-VL-2B-Instruct-full-trained.json', 'Qwen2-VL-7B-Instruct-1000-trained.json', 'Qwen2-VL-7B-Instruct-10000-trained.json', 'Qwen2-VL-7B-Instruct-base.json', 'Qwen2-VL-7B-Instruct-full-trained.json', 'Qwen25-VL-7B-Instruct-1000-trained.json', 'Qwen25-VL-7B-Instruct-10000-trained.json', 'Qwen25-VL-7B-Instruct-base.json', 'Qwen25-VL-7B-Instruct-full-trained.json']
20


In [6]:
for item in raw_test_dataset:
    print(json.dumps(item, indent=4))
    break

{
    "id": 6730,
    "image": {
        "file_name": "COCO_train2014_000000006730.jpg",
        "id": 6730,
        "coco_url": "http://images.cocodataset.org/train2014/COCO_train2014_000000006730.jpg",
        "image_width": 640,
        "image_height": 480,
        "date_captured": "2013-11-17 17:29:11",
        "flickr_url": "http://farm1.staticflickr.com/75/195448192_af8971813f_z.jpg"
    },
    "captions": [
        {
            "caption": "A cat sitting beside a wicker suitcase in front of some clothes. ",
            "image_id": 6730,
            "sle_score": 2.9013736248016357,
            "smog_index": 3.1291,
            "flesch_reading_ease": 81.85500000000002,
            "flesch_kincaid_grade": 4.823333333333334,
            "coleman_liau_index": 6.866666666666667,
            "automated_readability_index": 5.372500000000002,
            "dale_chall_readability_score": 6.863366666666667,
            "gunning_fog": 4.800000000000001,
            "syllable_count": 16,
    

In [7]:
class Evaluator:
    def __init__(self) -> None:
        self.tokenizer = PTBTokenizer()
        self.scorer_list = [
            # (Bleu(4), ["Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4"]),
            # (Meteor(), "METEOR"),
            # (Rouge(), "ROUGE_L"),
            # (Cider(), "CIDEr"),
            (Spice(), "SPICE"),
        ]
        self.evaluation_report = {}

    def score(self, golden_reference, candidate_reference):
        golden_reference = self.tokenizer.tokenize(golden_reference)
        candidate_reference = self.tokenizer.tokenize(candidate_reference)

        for scorer, method in self.scorer_list:
            score, scores = scorer.compute_score(golden_reference, candidate_reference)
            if isinstance(method, list):
                for sc, scs, m in zip(score, scores, method):
                    self.evaluation_report[m] = sc
            else:
                self.evaluation_report[method] = score

        return self.evaluation_report


golden_reference = [
    "The quick brown fox jumps over the lazy dog.",
    "The brown fox quickly jumps over the lazy dog.",
    "A sly brown fox jumps over the lethargic dog.",
    "The speedy brown fox leaps over the sleepy hound.",
    "A fast, brown fox jumps over the lazy dog.",
]
golden_reference = {k: [{'caption': v}] for k, v in enumerate(golden_reference)}

print(golden_reference)
# candidate_reference = [
#     "A fast brown fox leaps above the tired dog.",
#     "A quick brown fox jumps over the sleepy dog.",
#     "The fast brown fox jumps over the lazy dog.",
#     "The brown fox jumps swiftly over the lazy dog.",
#     "A speedy brown fox leaps over the drowsy dog.",
# ]
# candidate_reference = {k: [{'caption': v}] for k, v in enumerate(candidate_reference)}
# evaluator = Evaluator()
# evaluator.score(golden_reference, candidate_reference)
# print(evaluator.evaluation_report)

{0: [{'caption': 'The quick brown fox jumps over the lazy dog.'}], 1: [{'caption': 'The brown fox quickly jumps over the lazy dog.'}], 2: [{'caption': 'A sly brown fox jumps over the lethargic dog.'}], 3: [{'caption': 'The speedy brown fox leaps over the sleepy hound.'}], 4: [{'caption': 'A fast, brown fox jumps over the lazy dog.'}]}


In [8]:
def evaluate_captions(ground_truth_data, generated_results, output_file=None):
    print(len(ground_truth_data), len(generated_results))
    unique_image_ids = {}
    current_index = 0

    gt_data = {}

    annotation_id = 0
    for gt_item in ground_truth_data:
        image_id_hash = f'{gt_item['benchmark']}_{gt_item["image"]["id"]}'

        if unique_image_ids.get(image_id_hash) is None:
            unique_image_ids[image_id_hash] = annotation_id
            annotation_id += 1

    for gt_item in ground_truth_data:
        image_id_hash = f'{gt_item['benchmark']}_{gt_item["image"]["id"]}'

        if unique_image_ids[image_id_hash] not in gt_data:
            gt_data[unique_image_ids[image_id_hash]] = []

        for caption in gt_item["captions"]:
            gt_data[unique_image_ids[image_id_hash]].append({
                "caption": caption["caption"],
            })

    results = {}
    for result in generated_results:
        image_id_hash = f'{result['benchmark']}_{result["id"]}'

        if unique_image_ids[image_id_hash] not in results:
            results[unique_image_ids[image_id_hash]] = []

        if "trained_model_output" in result:
            results[unique_image_ids[image_id_hash]].append({"caption": result["trained_model_output"]})
        else:
            results[unique_image_ids[image_id_hash]].append({"caption": result["base_model_output"]})

    # print(gt_data.keys(), results.keys())
    print("started evaluation")
    evaluator = Evaluator()
    evaluator.score(gt_data, results)
    print(evaluator.evaluation_report)

    # with open(output_file, 'w') as of:
    #     json.dump(evaluator.evaluation_report, of, indent=5)

In [ ]:
for item in generated_files:
    expt_name = item.split('.')[0]
    print(item)
    with open(os.path.join(SCORED_CAPTION_FOLDER, item), 'r', encoding='utf-8') as f:
        expt_gen_caption = json.load(f)

        for i in range(len(expt_gen_caption)//1000):
            start = i*1000
            evaluate_captions(raw_test_dataset[start:start+1000], expt_gen_caption[start:start+1000], f'../eval/{item}')

        # for gen_caption in expt_gen_caption:
        #     if 'trained_model_output' in gen_caption:
        #         gen_caption['model_output'] = gen_caption['trained_model_output']
        #     else:
        #         gen_caption['model_output'] = gen_caption['base_model_output']

    #         gen_caption['sle_score'] = scorer.score([gen_caption['model_output']])
    #
    # with open(os.path.join(SCORED_CAPTION_FOLDER, item), 'w', encoding='utf-8') as f:
    #     json.dump(expt_gen_caption, f, indent=4)

gemma3-3-epoch-1000-trained.json
1000 1000
started evaluation


In [16]:
from sle import scorer

for item in generated_files:
    expt_name = item.split('.')[0]
    print(item)

    sle_min = 100000000
    sle_max = -100000000
    sle_sum = 0

    with open(os.path.join(SCORED_CAPTION_FOLDER, item), 'r', encoding='utf-8') as f:
        expt_gen_caption = json.load(f)

        for gen_caption in expt_gen_caption:
            if 'trained_model_output' in gen_caption:
                gen_caption['model_output'] = gen_caption['trained_model_output']
            else:
                gen_caption['model_output'] = gen_caption['base_model_output']

            # gen_caption['sle_score'] = scorer.score([gen_caption['model_output']])

            # print(json.dumps(gen_caption, indent=4))
            sle_min = min(sle_min, gen_caption['sle_score']['sle'][0])
            sle_max = max(sle_max, gen_caption['sle_score']['sle'][0])
            sle_sum = sle_sum + gen_caption['sle_score']['sle'][0]

        print(sle_sum/len(expt_gen_caption), sle_min, sle_max)

gemma3-3-epoch-1000-trained.json
1.1378024529210644 -0.7488585710525513 3.429227113723755
gemma3-3-epoch-10000-trained.json
3.0016828462361107 0.37884214520454407 4.400500774383545
gemma3-3-epoch-full-trained.json
3.1486854667121382 1.224833607673645 4.390599727630615
gemma3-base.json
0.9174876021315579 -0.8092108368873596 3.074036121368408
Llama-3.2-11B-Vision-base.json
1.1374136840608589 -0.9900307655334473 4.417754650115967
Llama-32-11B-Vision-1000-trained.json
1.6513029411647748 -1.0121078491210938 4.444007396697998
Llama-32-11B-Vision-10000-trained.json
4.1516657974923055 0.9225879907608032 4.446095943450928
Llama-32-11B-Vision-full-trained.json
4.035179381753682 0.6422208547592163 4.447038173675537
Qwen2-VL-2B-Instruct-1000-trained.json
2.184493473538863 -1.0540403127670288 4.442518711090088
Qwen2-VL-2B-Instruct-10000-trained.json
3.9805369587588486 -0.34708043932914734 4.4458794593811035
Qwen2-VL-2B-Instruct-base.json
2.184797964594719 -1.0415109395980835 4.441749095916748
Qwen2

In [17]:
! pip install lens-metric

     ---------------------------------------- 0.0/5.2 MB ? eta -:--:--
     ---- ----------------------------------- 0.5/5.2 MB 5.5 MB/s eta 0:00:01
     ------------------ --------------------- 2.4/5.2 MB 8.8 MB/s eta 0:00:01
     ---------------------------- ----------- 3.7/5.2 MB 7.6 MB/s eta 0:00:01
     -------------------------------------- - 5.0/5.2 MB 7.2 MB/s eta 0:00:01
     ---------------------------------------- 5.2/5.2 MB 7.0 MB/s eta 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: still running...
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/727.7 kB ? eta -:--:--
   --------------------------------------- 727.7/727.7 kB 22.7 MB/s eta 0:00:00
Us

  error: subprocess-exited-with-error
  
  × Building wheel for pandas (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [2527 lines of output]
      <string>:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
      C:\Users\pranon\AppData\Local\Temp\pip-build-env-ig167ugu\overlay\Lib\site-packages\setuptools\dist.py:759: SetuptoolsDeprecationWarning: License classifiers are deprecated.
      !!
      
              ********************************************************************************
              Please consider removing the following classifiers in favor of a SPDX license expression:
      
              License :: OSI Approved :: BSD License
      
              See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for details.
        

In [19]:
from lens import download_model, LENS_SALSA

lens_salsa_path = download_model("davidheineman/lens-salsa")
lens_salsa = LENS_SALSA(lens_salsa_path)

complex = [
    "They are culturally akin to the coastal peoples of Papua New Guinea."
]
simple = [
    "They are culturally similar to the people of Papua New Guinea."
]

scores, word_level_scores = lens_salsa.score(complex, simple, batch_size=8, devices=[0])
print(scores) # [72.40909337997437]

# LENS-SALSA also returns an error-identification tagging, recover_output() will return the tagged output
tagged_output = lens_salsa.recover_output(word_level_scores, threshold=0.5)
print(tagged_output)

ModuleNotFoundError: No module named 'lens'